# 06 · Information Extraction Baseline (RQ2)

A rule-based extractor for fees and required documents, scored field by field against the hand-labelled Phase 2 facts. S001 was used while writing the rules (development); S003 was not (held out), so S003 is the honest estimate.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'ml-service'))
import pandas as pd
import matplotlib.pyplot as plt
from chlatvei_ml.data import Dataset
pd.set_option('display.width', 160)
RESULTS = Path('..') / 'data' / 'evaluation' / 'results'
# Chart style: categorical palette validated for colour-blind separation (see docs/data-science/phase6_report.md)
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300']
SURFACE, INK, MUTED = '#fcfcfb', '#0b0b0b', '#52514e'
plt.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'axes.edgecolor': MUTED, 'axes.labelcolor': INK,
                     'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'font.size': 10})
ds = Dataset.load()

In [2]:
from chlatvei_ml.extraction import extract, evaluate_source
for sid in ['S001', 'S003']:
    r = extract(ds.source_text(sid))
    print(sid, 'fees found:', sorted({f['amount'] for f in r.fees}))
    print(sid, 'documents found:', r.documents, '\n')

S001 fees found: [500, 2000, 20000, 30000, 90000, 95000, 100000, 120000, 180000]
S001 documents found: ['Copy of Khmer Nationality Identity Card Certified by the Competent Authority', '3 white 4x6 photos', 'Medical Certificate', "Old driver's license (for duplicate and triple exchange)", 'Certificate of driving school education (for driving test, type C, D and E).', 'Passport', 'Valid visa', 'Certificate of current residence issued by the competent authority', "Your national driver's license, which is valid in the case of non-English or French, must be translated into English or Khmer from the embassy of your country (for a driver's license)"] 

S003 fees found: [38000, 40000, 48000, 52000, 65000, 67000, 70000, 101000, 120000, 125000, 150000, 180000, 240000, 300000]
S003 documents found: ['បាត់បណ្ណសម្គាល់រថយន្តតើត្រូវការឯកសារអ្វីខ្លះដើម្បីធ្វើថ្មី និងត្រូវទៅធ្វើនៅទីណា?', 'ចុះឈ្មោះបង្កើតគណនីចុះបញ្ជីយានយន្តតាមប្រព័ន្ធ Online តើអាចទិញផ្លាកលេខដោយដាក់ឈ្មោះអ្នកផ្សេងបានដែរឬទេ?', 'មិនអាចទិញផ្ល

In [3]:
pd.read_csv(RESULTS / 'extraction.csv')

,source_id,field,tp,predicted,gold,precision,recall,f1,split
0,S001,fee,5,9,5,0.556,1.000,0.714,development
1,S001,required_document,8,9,9,0.667,0.889,0.762,development
2,S003,fee,2,14,2,0.143,1.000,0.250,held-out
3,S003,required_document,0,4,4,0.000,0.000,0.000,held-out


## Error analysis
- **Fees, high recall / low precision:** the regex finds every amount, including fines (500 ៛/day) and fees outside the annotation scope (types C–E, plate prices). Part of the 'error' is incomplete gold data: precision is a lower bound.
- **Documents on S003: 0 found.** The vehicle page states documents inside a sentence ("…attachments of your vehicle's receipt of import tax, letter from the dealer…"), not as a list. Rules built on S001's list layout do not transfer.
- **Khmer FAQ content** (age limits, validity, test stages) is prose and is not extracted at all.

**Conclusion for RQ2:** layout rules are a usable baseline for list-formatted English pages only. Prose and Khmer need a learned or LLM-assisted extractor, evaluated on this same gold data, and every extraction still goes to admin review.